In [1]:
%pip install -q transformers datasets evaluate trl peft

Note: you may need to restart the kernel to use updated packages.


In [9]:
from datasets import load_dataset

dataset = load_dataset("kuotient/orca-math-korean-dpo-pairs")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['system', 'question', 'chosen', 'rejected'],
        num_rows: 192848
    })
})

In [10]:
dataset['train'][0]

{'system': '당신은 인공지능 어시스턴트입니다.',
 'question': '정국이 5위입니다. 정국보다 결승선을 먼저 통과한 사람의 수를 찾아보세요.',
 'chosen': '정국이 5위라면 4명이 정국보다 먼저 결승선을 통과한 셈입니다.',
 'rejected': ' 화물 열차의 속도를 구하려면 먼저 두 열차 사이의 상대 속도를 구해야 합니다. 두 열차가 서로 반대 방향으로 이동하기 때문에 두 열차의 속도를 더하여 상대 속도를 구합니다.\n\n남자의 기차는 시속 50km로 달리고 있습니다. 화물 열차의 속도를 \\( v \\) km/h로 표시해 봅시다.\n\n두 열차가 서로 지나갈 때 두 열차의 길이를 합친 길이를 통과합니다. 화물 열차의 길이는 280m로 주어집니다.\n\n두 열차가 서로 지나갈 때 이동한 총 거리는 두 열차의 길이의 합으로, 50m(남자의 열차 길이) + 280m(화물 열차 길이) = 330m입니다.\n\n두 열차가 서로 지나가는 데 걸리는 시간은 9초로 주어집니다.\n\n상대 속도(\\( v_{rel} \\))는 다음 공식을 사용하여 계산할 수 있습니다:\n\\[ v_{rel} = \\frac{Distance}{Time} \\]\n\n주어진 값을 대입하면 다음과 같습니다:\n\\[ v_{rel} = \\frac{330 \\text{ m}}{9 \\text{ s}} \\]\n\n\\[ v_{rel} = 36.67 \\text{ m/s} \\]\n\n이제 상대 속도를 km/h로 변환하여 주어진 속도 단위와 일치시켜야 합니다. 1m/s는 3.6km/h와 같으므로 다음과 같이 쓸 수 있습니다:\n\\[ v_{rel} = 36.67 \\text{ m/s} \\times \\frac{3.6 \\text{ km/h}}{1 \\text{ m/s}} \\]\n\n\\[ v_{rel} = 132.28 \\text{ km/h} \\]\n\n상대 속도는 두 열차의 속도를 더한 것이므로 화물 열차의 속도(\\( v \\))는 상대 속도에서 남자의 열차

In [11]:
splitted_dataset = dataset['train'].train_test_split(test_size=0.01)
splitted_dataset

DatasetDict({
    train: Dataset({
        features: ['system', 'question', 'chosen', 'rejected'],
        num_rows: 190919
    })
    test: Dataset({
        features: ['system', 'question', 'chosen', 'rejected'],
        num_rows: 1929
    })
})

In [12]:
splitted_dataset.save_to_disk("data/math-korean-dpo")

Saving the dataset (0/1 shards):   0%|          | 0/190919 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1929 [00:00<?, ? examples/s]

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('unsloth/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

In [15]:
tokenizer.save_pretrained('model/Meta-Llama-3.1-8B-Instruct')

('model/Meta-Llama-3.1-8B-Instruct/tokenizer_config.json',
 'model/Meta-Llama-3.1-8B-Instruct/special_tokens_map.json',
 'model/Meta-Llama-3.1-8B-Instruct/tokenizer.json')

In [13]:
def prepare_sample_text(example):
    """Prepare the text from a sample of the dataset."""
    text = f"SYSTEM: {example['system']}\n\nQuestion: {example['question']}\n\nAnswer: {example['chosen']}"
    return text

In [17]:
from tqdm import tqdm
def chars_token_ratio(dataset, tokenizer, nb_examples=400):
    """
    Estimate the average number of characters per token in the dataset.
    """
    total_characters, total_tokens = 0, 0
    for _, example in tqdm(zip(range(nb_examples), iter(dataset)), total=nb_examples):
        text = prepare_sample_text(example)
        total_characters += len(text)
        if tokenizer.is_fast:
            total_tokens += len(tokenizer(text).tokens())
        else:
            total_tokens += len(tokenizer.tokenize(text))

    return total_characters / total_tokens

In [18]:
chars_token_ratio(splitted_dataset['train'], tokenizer)

100%|██████████| 400/400 [00:00<00:00, 682.55it/s]


1.6758690188857648

In [4]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    'unsloth/Meta-Llama-3.1-8B-Instruct', 
    trust_remote_code=True, 
    cache_dir='model/Meta-Llama-3.1-8B-Instruct',
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [5]:
model.save_pretrained('model/Meta-Llama-3.1-8B-Instruct')

In [9]:
from transformers import TrainingArguments
training_args = TrainingArguments(output_dir="./")
training_args.set_deepspeed({"abc": "abc"})
training_args

AttributeError: 'TrainingArguments' object has no attribute 'set_deepspeed'